In [26]:
import numpy as np
import pandas as pd
import torch
from torch import nn

data = pd.read_csv('train.csv')

In [27]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [28]:
data = data.to_numpy()
# 0th col are the labels of the images
# 1st col up to 784th col are pixel values

In [29]:
data

array([[1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [7, 0, 0, ..., 0, 0, 0],
       [6, 0, 0, ..., 0, 0, 0],
       [9, 0, 0, ..., 0, 0, 0]], shape=(42000, 785))

In [30]:
labels = data[:, 0]
data = data[:, 1:785]

In [31]:
data.shape

(42000, 784)

In [32]:
train_split = int(0.8 * len(data))
y_train, y_test = labels[:train_split], labels[train_split:]
X_train, X_test = data[:train_split], data[train_split:]

In [62]:
X_train, X_test = X_train.to(torch.float32), X_test.to(torch.float32)
y_train, y_test = y_train.to(torch.int64), y_test.to(torch.int64)

In [63]:
class NumClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 10)
        self.layer2 = nn.Linear(10, 10)
        self.layer3 = nn.Linear(10,10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        return x

In [64]:
model_1 = NumClassificationModel()

In [65]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model_1.parameters(), lr = 0.1)

In [66]:
X_train.shape

torch.Size([33600, 784])

In [67]:
y_logits = model_1(X_train)
print(y_logits, y_logits.shape)
y_pred = nn.Softmax(dim=1)(y_logits)
print(y_pred, y_pred.shape)
y_pred_labels = torch.argmax(y_pred, dim=1)
print(y_pred_labels, y_pred_labels.shape)

tensor([[ -9.7019, -10.5251,  -2.5857,  ...,  22.4416,  -9.1976,   1.8206],
        [-15.1033, -27.0865, -15.5697,  ...,  33.1956, -17.0515,   9.0244],
        [ -7.6811,  -2.8395,  -3.5848,  ...,   9.8248,  -1.9581,  -1.6101],
        ...,
        [-17.3600, -30.9414, -14.0012,  ...,  33.5140,  -7.2692,   6.2876],
        [ -5.0283, -10.1170,  -6.3694,  ...,  11.8225,  -6.2744,   4.1178],
        [-10.2445, -15.9690,  -9.1574,  ...,  19.8017, -10.8509,   2.2352]],
       grad_fn=<AddmmBackward0>) torch.Size([33600, 10])
tensor([[1.0971e-14, 4.8168e-15, 1.3513e-11,  ..., 9.9999e-01, 1.8166e-14,
         1.1076e-09],
        [1.2756e-23, 7.9704e-29, 8.0013e-24,  ..., 1.2070e-02, 1.8182e-24,
         3.8392e-13],
        [2.1764e-08, 2.7570e-06, 1.3084e-06,  ..., 8.7186e-01, 6.6562e-06,
         9.4268e-06],
        ...,
        [4.2253e-23, 5.3399e-29, 1.2151e-21,  ..., 5.2499e-01, 1.0192e-18,
         7.8687e-13],
        [3.3802e-09, 2.0843e-11, 8.8410e-10,  ..., 7.0332e-02, 9.7221e-1

In [68]:
def accuracy(actual_labels, pred_labels):
    '''
    both actual_labels and pred_labels are vectors where each entry is an example
    '''
    count = 0
    for i in actual_labels:
        if actual_labels[i] == pred_labels[i]:
            count += 1
    return count / len(actual_labels) * 100

In [70]:
# training loop
epochs = 1000 

for epoch in range(epochs):
    model_1.train()
    
    # forward pass
    y_logits = model_1(X_train)
    y_pred = nn.Softmax(dim=1)(y_logits)
    y_pred_labels = torch.argmax(y_pred, dim=1)

    # calculate the loss
    loss = loss_fn(y_logits, y_train)
    acc = accuracy(y_train, y_pred_labels)

    # optmizier.zero_grad()
    optimizer.zero_grad()

    # loss.backward()
    loss.backward()

    # optimizer.step()
    optimizer.step()

    # testing loop
    model_1.eval()
    with torch.inference_mode():
        # forward pass
        y_test_logits = model_1(X_test)
        y_test_pred = nn.Softmax(dim=1)(y_logits)
        y_test_pred_labels = torch.argmax(y_test_pred, dim=1)

        # calculate the loss
        test_loss = loss_fn(y_test_logits, y_test)
        test_acc = accuracy(y_test, y_test_pred_labels)

        if epoch % 100 == 0: 
            print(f'epoch: {epoch} | train loss: {loss:.5f} | train acc: {acc}% | test loss: {test_loss:.5f} | test acc: {test_acc}%')


epoch: 0 | train loss: 2.30847 | train acc: 10.324404761904763% | test loss: 2.30856 | test acc: 0.0%


KeyboardInterrupt: 